# Sequence Inference Demo

Ce notebook reprend le script `sequence_inference_demo.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Smoke test d'inference sequence sur video/features pour verifier le chemin live minimal.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Run sequence danger inference on a clip using precomputed pose features.
- Artefacts controles : Precomputed-pose inference smoke test exists. (`runs/exp_018_inference_smoke_tcn_aug/inference_config.json`); Raw-video inference smoke test exists. (`runs/exp_019_raw_video_inference_smoke_tcn_aug/inference_config.json`).
- Le script ecrit ou lit des artefacts experimentaux dans `runs/`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_inference_demo.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from ml_pipeline import ROOT, YOLO_POSE_WEIGHTS, draw_overlay, load_dataset, pose_rows_for_results, read_frame, write_json, zone_polygon
from sequence_experiments import add_frame_motion_features, make_model


## Fonction `load_checkpoint`

Cette cellule definit `load_checkpoint`. Elle prepare une partie du script.

In [ ]:
def load_checkpoint(model_path, device):
    payload = torch.load(model_path, map_location=device, weights_only=False)
    model = make_model(payload["kind"], int(payload["seq_len"]), int(payload["input_dim"]), len(payload["horizons"]))
    model.load_state_dict(payload["state_dict"])
    model.to(device)
    model.eval()
    return model, payload


## Fonction `build_sequences`

Cette cellule definit `build_sequences`. Elle prepare une partie du script.

In [ ]:
def build_sequences(pose, feature_cols, mean, std, seq_len):
    pose = add_frame_motion_features(pose.copy())
    for col in feature_cols:
        if col not in pose:
            pose[col] = 0.0
    values = pose[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
    values = ((values - mean.reshape(1, -1)) / std.reshape(1, -1)).astype(np.float32)
    X = np.zeros((len(values), seq_len, len(feature_cols)), dtype=np.float32)
    for idx in range(len(values)):
        start = max(0, idx - seq_len + 1)
        seq = values[start : idx + 1]
        if len(seq) < seq_len:
            pad = np.repeat(seq[:1], seq_len - len(seq), axis=0)
            seq = np.vstack([pad, seq])
        X[idx] = seq[-seq_len:]
    return X


## Fonction `extract_pose_from_raw_video`

Cette cellule definit `extract_pose_from_raw_video`. Elle prepare une partie du script.

In [ ]:
def extract_pose_from_raw_video(video_path, polygon, imgsz, pose_conf, pose_batch):
    from ultralytics import YOLO

    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    model = YOLO(str(YOLO_POSE_WEIGHTS))
    video = {
        "video_id": video_path.stem,
        "path": str(video_path),
        "fps": fps,
        "width": width,
        "height": height,
    }
    rows = []
    frames = []
    indices = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
        indices.append(frame_idx)
        if len(frames) >= pose_batch:
            results = model.predict(frames, imgsz=imgsz, conf=pose_conf, verbose=False)
            rows.extend(pose_rows_for_results(video, "inference", indices, results, polygon))
            frames = []
            indices = []
        frame_idx += 1
    if frames:
        results = model.predict(frames, imgsz=imgsz, conf=pose_conf, verbose=False)
        rows.extend(pose_rows_for_results(video, "inference", indices, results, polygon))
    cap.release()
    return pd.DataFrame(rows)


## Fonction `predict_batches`

Cette cellule definit `predict_batches`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_batches(model, X, device, batch_size):
    probs = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start : start + batch_size]).to(device)
        probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.vstack(probs)


## Fonction `alarm_state`

Cette cellule definit `alarm_state`. Elle prepare une partie du script.

In [ ]:
def alarm_state(times, risks, threshold, persistence_frames):
    states = []
    run = 0
    for risk in risks:
        if risk >= threshold:
            run += 1
        else:
            run = 0
        states.append(int(run >= persistence_frames))
    return states


## Fonction `make_contact_sheet`

Cette cellule definit `make_contact_sheet`. Elle prepare une partie du script.

In [ ]:
def make_contact_sheet(out_dir, video_path, pred, polygon, risk_col, threshold):
    top = pred.sort_values(risk_col, ascending=False).head(12)
    tiles = []
    for _, row in top.iterrows():
        frame = read_frame(video_path, int(row["frame"]))
        if frame is None:
            continue
        text = [
            f"{risk_col}={float(row[risk_col]):.3f}",
            f"thr={threshold:.2f} alarm={int(row['alarm'])}",
            f"t={float(row['time_s']):.2f}s",
        ]
        out = draw_overlay(frame, polygon, text)
        out = cv2.resize(out, (320, 180))
        tiles.append(out)
    if not tiles:
        return
    rows = []
    for i in range(0, len(tiles), 3):
        chunk = tiles[i : i + 3]
        while len(chunk) < 3:
            chunk.append(np.zeros_like(tiles[0]))
        rows.append(np.hstack(chunk))
    cv2.imwrite(str(out_dir / "top_risk_contact_sheet.jpg"), np.vstack(rows))


## Fonction `write_annotated_video`

Cette cellule definit `write_annotated_video`. Elle prepare une partie du script.

In [ ]:
def write_annotated_video(out_path, video_path, pred, polygon, risk_col, threshold):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))
    pred_by_frame = {int(row["frame"]): row for _, row in pred.iterrows()}
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        row = pred_by_frame.get(frame_idx)
        if row is not None:
            risk = float(row[risk_col])
            alarm = int(row["alarm"])
            text = [f"{risk_col}={risk:.3f}", f"thr={threshold:.2f}", "ALARM" if alarm else "clear"]
            frame = draw_overlay(frame, polygon, text)
        writer.write(frame)
        frame_idx += 1
    cap.release()
    writer.release()


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    sequence_run = Path(args.sequence_run)
    if not sequence_run.is_absolute():
        sequence_run = ROOT / sequence_run
    model_path = Path(args.model)
    if not model_path.is_absolute():
        model_path = sequence_run / "models" / model_path
    out_dir = Path(args.out_dir)
    if not out_dir.is_absolute():
        out_dir = ROOT / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    model, payload = load_checkpoint(model_path, device)
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    feature_payload = json.loads((sequence_run / "features" / "sequence_feature_columns.json").read_text(encoding="utf-8"))
    feature_cols = feature_payload["feature_columns"]
    videos, _, _, zones = load_dataset()
    polygon = zone_polygon(zones)
    if args.raw_video:
        raw_path = Path(args.raw_video)
        if not raw_path.is_absolute():
            raw_path = ROOT / raw_path
        pose = extract_pose_from_raw_video(raw_path, polygon, args.imgsz, args.pose_conf, args.pose_batch)
        if args.video_id:
            pose["video_id"] = args.video_id
    else:
        pose_path = Path(args.pose_features)
        if not pose_path.is_absolute():
            pose_path = ROOT / pose_path
        pose = pd.read_csv(pose_path)
        if args.video_id:
            pose = pose[pose["video_id"] == args.video_id].copy()
        elif args.video_path:
            pose = pose[pose["path"].astype(str).str.replace("\\\\", "/") == args.video_path.replace("\\", "/")].copy()
        else:
            raise SystemExit("--video-id, --video-path, or --raw-video is required")
    if pose.empty:
        raise SystemExit("No pose rows matched the requested video")
    pose = pose.sort_values("frame").reset_index(drop=True)
    X = build_sequences(pose, feature_cols, mean, std, int(payload["seq_len"]))
    probs = predict_batches(model, X, device, args.batch_size)
    pred = pose[["video_id", "path", "frame", "time_s", "fps"]].copy()
    for idx, horizon in enumerate(payload["horizons"]):
        pred[f"risk_{float(horizon):.1f}s"] = probs[:, idx]
    risk_col = f"risk_{args.horizon:.1f}s"
    pred["alarm"] = alarm_state(pred["time_s"], pred[risk_col], args.threshold, args.persistence_frames)
    pred.to_csv(out_dir / "risk_predictions.csv", index=False)

    source_path = Path(str(pose["path"].iloc[0]))
    video_path = source_path if source_path.is_absolute() else ROOT / source_path
    make_contact_sheet(out_dir, video_path, pred, polygon, risk_col, args.threshold)
    if args.write_video:
        write_annotated_video(out_dir / "annotated_risk.mp4", video_path, pred, polygon, risk_col, args.threshold)
    write_json(
        out_dir / "inference_config.json",
        {
            "sequence_run": str(sequence_run),
            "model": str(model_path),
            "video_id": str(pose["video_id"].iloc[0]),
            "video_path": str(video_path),
            "raw_video_mode": bool(args.raw_video),
            "horizon": args.horizon,
            "threshold": args.threshold,
            "persistence_frames": args.persistence_frames,
            "rows": int(len(pred)),
            "max_risk": float(pred[risk_col].max()),
            "first_alarm_time_s": None if pred[pred["alarm"] == 1].empty else float(pred[pred["alarm"] == 1]["time_s"].iloc[0]),
        },
    )
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Run sequence danger inference on a clip using precomputed pose features.")
    parser.add_argument("--sequence-run", default="runs/exp_010_sequence_len60_focal_catalogue")
    parser.add_argument("--model", default="tcn_aug.pt")
    parser.add_argument("--pose-features", default="runs/exp_006_final_research/features/pose_features.csv")
    parser.add_argument("--video-id", default=None)
    parser.add_argument("--video-path", default=None)
    parser.add_argument("--raw-video", default=None)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--horizon", type=float, default=1.0)
    parser.add_argument("--threshold", type=float, default=0.35)
    parser.add_argument("--persistence-frames", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--pose-conf", type=float, default=0.10)
    parser.add_argument("--pose-batch", type=int, default=16)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--write-video", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement concret du smoke test d'inference sequence
# Cette configuration utilise les features de pose deja calculees pour une video connue.
from datetime import datetime
import sys

OUT_DIR = f"runs/exp_018_inference_smoke_tcn_aug_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = [
    "--video-id", "captures_ilyas_unsafe_unsafe_blooza_good_20260512_190912_fdc9750f",
    "--out-dir", OUT_DIR,
]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_inference_demo.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
